# Dataset

- **Police Press releases about traffic accidents in Malta: local_news_articles.csv** <br>
- **Text of local news articles covering accidents: police_press_releases.csv** <br>
- **date.Nager API for Malta National Holidays** <br>
- **Open-meteo API for weather forecasts** <br>


In [113]:
# Imports
import pandas as pd
from collections import Counter
import requests
from datetime import datetime
from sklearn.feature_extraction.text import TfidfVectorizer #Vecctorizing Text
from sklearn.preprocessing import StandardScaler #Normalizing values in the weather API 0-1

In [121]:
# Get all days which were holidays in Malta via date.nager API - This was done after merging the two datasets, however code wise it must be done before slicing the 'date published' for both datasets
# to compare the date_published with the holiday date, such that I can then do a boolean column for when a crash occured during a holiday (Comparison).

year = 2025 #Dummy year could be any year, used for URL since documentation asks for it
country = 'MT'
holiday_dates = [] # This is to be used later on by both datasets.

holiday_url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country}"
holidays = requests.get(holiday_url).json()

In [115]:
# Similarily to the above API, to get weather forecast data - it must be retrieved before slicing the 'date published' column later on.
# Malta latitude: 35.917973 , longitude 14.409943 - latlong.net/place/malta-11532.html
longitude = 14.409943
latitude = 35.917973
start_date = '2024-12-07' # Oldest date in merging_dataset , found using excel filtering
end_date = '2025-10-31' # Most recent date in merging_dataset

#  https://open-meteo.com/en/docs
forecast_url = f"https://archive-api.open-meteo.com/v1/era5?latitude={latitude}&longitude={longitude}&start_date={start_date}&end_date={end_date}&daily=temperature_2m_mean,apparent_temperature_mean,precipitation_sum,precipitation_hours,precipitation_probability_mean,wind_speed_10m_max&timezone=Europe/Malta"

forecasts = requests.get(forecast_url).json()

print(forecasts)
#MUST NORMALIZE THESE VALUES

{'latitude': 35.88752, 'longitude': 14.418605, 'generationtime_ms': 56.30636215209961, 'utc_offset_seconds': 3600, 'timezone': 'Europe/Malta', 'timezone_abbreviation': 'GMT+1', 'elevation': 36.0, 'daily_units': {'time': 'iso8601', 'temperature_2m_mean': '°C', 'apparent_temperature_mean': '°C', 'precipitation_sum': 'mm', 'precipitation_hours': 'h', 'precipitation_probability_mean': 'undefined', 'wind_speed_10m_max': 'km/h'}, 'daily': {'time': ['2024-12-07', '2024-12-08', '2024-12-09', '2024-12-10', '2024-12-11', '2024-12-12', '2024-12-13', '2024-12-14', '2024-12-15', '2024-12-16', '2024-12-17', '2024-12-18', '2024-12-19', '2024-12-20', '2024-12-21', '2024-12-22', '2024-12-23', '2024-12-24', '2024-12-25', '2024-12-26', '2024-12-27', '2024-12-28', '2024-12-29', '2024-12-30', '2024-12-31', '2025-01-01', '2025-01-02', '2025-01-03', '2025-01-04', '2025-01-05', '2025-01-06', '2025-01-07', '2025-01-08', '2025-01-09', '2025-01-10', '2025-01-11', '2025-01-12', '2025-01-13', '2025-01-14', '2025-0

In [116]:
# Police Press dataframe
police_press_df = pd.read_csv("Datasets\police_press_releases.csv")

police_press_df.info()
print('\n')
print(police_press_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   title           111 non-null    object
 1   date_published  111 non-null    object
 2   date_modified   111 non-null    object
 3   content         111 non-null    object
dtypes: object(4)
memory usage: 3.6+ KB


                                               title date_published  \
0  Collision between a car and a motorbike in Żur...     2025-10-09   
1                    Car-motorcycle traffic accident     2025-06-20   
2              Car-motorcycle collision in Ħal Qormi     2025-05-12   
3     Collision between motorcycle and car in Għaxaq     2025-07-30   
4                           Car-motorcycle collision     2025-04-07   

  date_modified                                            content  
0    2025-10-09  Today, at around 0930hrs, the Police were info...  
1    2025-06-20  Yesterda

In [ ]:
# Removal of date_modified column on original DF as it is beleived to not be useful.
# Traditional MLs can't use dates as is, will have to convert into YEAR, MONTH, DAY.

police_press_df.drop(columns=["date_modified"], inplace=True)

# Converting to dateTime and slicing date published into YEAR, MONTH, DAY
# Stripping any trailing spaces, blank spaces etc
police_press_df['date_published'] = police_press_df['date_published'].astype(str).str.strip()

# For weather API, creating a dataframe for the data imported from the API
daily_police_press = forecasts["daily"]   # from API Response
weather_df = pd.DataFrame({
    "date": pd.to_datetime(daily_police_press["time"]), # Date
    "temperature_2m_mean": daily_police_press["temperature_2m_mean"], # Avg daily air temperature at 2 meters above ground - Descr copy pasted from API
    "apparent_temperature_mean": daily_police_press["apparent_temperature_mean"], # Avg daily apparent temperature
    "precipitation_sum": daily_police_press["precipitation_sum"], # Sum of daily precipitation (including rain, showers and snowfall)
    "precipitation_hours": daily_police_press["precipitation_hours"], # The number of hours with rain
    #"precipitation_probability_mean": daily_police_press["precipitation_probability_mean"], # Probability of precipitation - ALTHOUGH USEFUL, THIS RETURNS NO DATA FOR ANY ROW
    "wind_speed_10m_max": daily_police_press["wind_speed_10m_max"] # Maximum wind speed on a day
})


police_press_df['date_published'] = (
    pd.to_datetime(
        police_press_df['date_published'].astype(str).str.strip(),
        format='mixed', # Was having trouble with the format even though all rows contain dd/mm/YYYY - specifying to mixed worked.
        dayfirst=True,  # Although specified format to mixed, specifying dayFirst = True should keep the data accurate.
        errors='coerce'
    )
)

# Filtering dates from weather_df to dates only present in plice_press df
police_press_df["date_only"] = police_press_df["date_published"].dt.date
weather_df["date_only"] = weather_df["date"].dt.date

# Merge only matching dates
merged_df_police_press = police_press_df.merge(
    weather_df,
    on="date_only",
    how="left"  # I can use INNER here in this case, but for assurance just got all via left
)

# Remove useless dates
merged_df_police_press.drop(columns=["date_only","date"], inplace=True)

# For Holidays API, Slicing out the year and keeping only Month-Day
holiday_dates = [h["date"][5:] for h in holidays]

# Adding new boolean column is_holiday, if date_published is on holiday > 1 else > 0
# ChatGPT Helped here with the comparison in terms of lambda expression
merged_df_police_press['is_holiday'] = merged_df_police_press['date_published'].apply(
    lambda d: 1 if d.strftime("%m-%d") in holiday_dates else 0
)

merged_df_police_press['year'] = merged_df_police_press['date_published'].dt.year
merged_df_police_press['month'] = merged_df_police_press['date_published'].dt.month
merged_df_police_press['day'] = merged_df_police_press['date_published'].dt.day

# Simply converting the columns into int to remove decimal
merged_df_police_press['year'] = merged_df_police_press['year'].astype(int)
merged_df_police_press['month'] = merged_df_police_press['month'].astype(int)
merged_df_police_press['day'] = merged_df_police_press['day'].astype(int)


#Dropping date_published column as it is no longer needed
merged_df_police_press.drop(columns=["date_published"], inplace=True)

#Cols to be normalized for the model
numeric_cols = [
    "temperature_2m_mean",
    "apparent_temperature_mean",
    "precipitation_sum",
    "precipitation_hours",
    #"precipitation_probability_mean",
    "wind_speed_10m_max"
]

'''
Fit and transform only the aforementioned numeric columns.
Negative values are completely normal since Z-Score works by centering data around 0, with STD > 1
Hence values above the mean > +
             below the mean > -
             equal the mean > 0
'''
scaler = StandardScaler()
merged_df_police_press[numeric_cols] = scaler.fit_transform(merged_df_police_press[numeric_cols])

merged_df_police_press.info()
print('\n')
print(merged_df_police_press.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 111 entries, 0 to 110
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   title                      111 non-null    object 
 1   content                    111 non-null    object 
 2   temperature_2m_mean        111 non-null    float64
 3   apparent_temperature_mean  111 non-null    float64
 4   precipitation_sum          111 non-null    float64
 5   precipitation_hours        111 non-null    float64
 6   wind_speed_10m_max         111 non-null    float64
 7   is_holiday                 111 non-null    int64  
 8   year                       111 non-null    int32  
 9   month                      111 non-null    int32  
 10  day                        111 non-null    int32  
dtypes: float64(5), int32(3), int64(1), object(2)
memory usage: 8.4+ KB


                                               title  \
0  Collision between a car and a moto

In [118]:
# As for tags which are in the other dataset, I have created a way of extracting the tags from the title/content that is found in this dataset.
# The help of chatGPT was utilized here, mostly in the creation of the assign_tags function.

tag_keywords = {
    'tag_National': ['national'],
    'tag_Accident': ['accident'],
    'tag_Traffic': ['traffic'],
    'tag_Police': ['police'],
    'tag_Transport': ['transport']
}

# Function to assign tags
def assign_tags(text):
    tags = {}
    text_lower = str(text).lower()
    for tag, keywords in tag_keywords.items():
        tags[tag] = int(any(word.lower() in text_lower for word in keywords))
    return pd.Series(tags)

all_text_series = merged_df_police_press['title'] + " " + merged_df_police_press['content']

# Apply tagging row by row
df_tags = all_text_series.apply(assign_tags)

# Assign the tag columns directly to the original DataFrame
for col in df_tags.columns:
    merged_df_police_press[col] = df_tags[col]

In [119]:
# Combining any content text columns for..

# Combine title and content into a single field, should not have any missing data but good practice
merged_df_police_press['content'] = merged_df_police_press['title'].fillna('') + " " + merged_df_police_press['content'].fillna('')

# Dropping title column
merged_df_police_press.drop(columns=['title'], inplace=True)

In [ ]:
# police_press_releases_cleaned as cleaned dataset for EDA & model training usage
output_file_path_one = 'Datasets\police_press_releases_cleaned.csv'

merged_df_police_press.to_csv(
    output_file_path_one, 
    index=False,
    encoding='utf-8'
)

print(f"Data exported successfully to: {output_file_path_one}")


Data exported successfully to: Datasets\police_press_releases_cleaned.csv


In [ ]:
# Local News Articles dataframe

local_news_df = pd.read_csv("Datasets\local_news_articles.csv")

local_news_df.info()
print('\n')
print(local_news_df.head())

In [ ]:
# Confirming only 2 source names are present, one-hot encoding
print('Sources:')
source_name = local_news_df['source_name'].unique()
print(source_name)

# Confirming multiple authors, will be used for the amount of times an author shows up - frequency encoding.
print('\nAuthors:')
author_names = local_news_df['author_name'].unique()
print(author_names)


# Confirming that categories is a completely empty column with empty dictionaries.
print('\nCategories Repr:')
local_news_df['categories'].apply(repr).head()

In [ ]:
# Dropping the columns which are deemed useless
# Removal of: article_id, url, source_url, created_at (publish_date is the relevant field),top_image_url, categories(confirmed to have no values)
local_news_df.drop(columns=["article_id" , "url", "source_url" , "created_at" , "top_image_url", "categories" , "source_name", "author_name"], inplace=True)

# One-hot encoding Source name
#local_news_df = pd.get_dummies(local_news_df, columns=["source_name"], dtype=int)
# This has been removed as it is not in police_press_releases dataset and it does not provide any relevant data - moreover it is heavily skewed towards 'Times of Malta'

# Frequency encoding authors
# author_counts = local_news_df["author_name"].value_counts()
# local_news_df["author_frequency"] = local_news_df['author_name'].map(author_counts)
# This is also not in police_press_releases datset and catering for this data (by for example placeing 0) would reduce the entire integrity of the dataset.
# Hence we agreed to focus on patterns not related to authors and to exclude them completely from the dataset

#Normalizing frequency for ML models.
#local_news_df["author_frequency"] = local_news_df["author_frequency"] / len(local_news_df)
# No longer needed

#Dropping original column
#local_news_df.drop(columns=["author_name"], inplace=True)

local_news_df.info()



In [ ]:
# Slicing the Date Published column into Year, Month, Day as done previously (Copy Pasted code)

local_news_df['publish_date'] = local_news_df['publish_date'].astype(str).str.strip()

# Convert holiday dates to MM-DD
holiday_dates = [h["date"][5:] for h in holidays]  # list of 'MM-DD'

# Adding new boolean column here is_holiday, if date_published is on holiday > 1 else > 0
local_news_df['is_holiday'] = local_news_df['publish_date'].apply(
    lambda d: 1 if d[5:] in holiday_dates else 0
)


local_news_df['publish_date'] = (
    pd.to_datetime(
        local_news_df['publish_date'].astype(str).str.strip(),
        format='mixed',
        dayfirst=True,
        errors='coerce'
    )
)

local_news_df['year'] = local_news_df['publish_date'].dt.year
local_news_df['month'] = local_news_df['publish_date'].dt.month
local_news_df['day'] = local_news_df['publish_date'].dt.day

local_news_df['year'] = local_news_df['year'].astype(int)
local_news_df['month'] = local_news_df['month'].astype(int)
local_news_df['day'] = local_news_df['day'].astype(int)

local_news_df.drop(columns=["publish_date"], inplace=True)

local_news_df.info()

In [ ]:
# For tags, I firstly turn the dicts into a single flattened list of all unique tags.
# The help of ChatGPT was utilized here.

# Step 1: Parse the tags manually
def parse_tags_manual(x):
    if not isinstance(x, str) or x.strip() == "":
        return []
    x = x.strip('{}')
    tags = [tag.strip().strip('"').strip("'") for tag in x.split(',')]
    return [tag for tag in tags if tag]

tags_lists = local_news_df['tags'].apply(parse_tags_manual)

# Step 2: Flatten all tags into a single list
all_tags = [tag for sublist in tags_lists for tag in sublist]

# Step 3: Count frequencies
tag_counts = Counter(all_tags)

# Step 4: Convert to a sorted DataFrame for readability
tag_freq_df = pd.DataFrame(tag_counts.items(), columns=['tag', 'frequency']).sort_values(by='frequency', ascending=False)

# Display the frequencies
print(tag_freq_df)

# Step 5: Get top 5 tags
top_tags = tag_freq_df['tag'].head(5).tolist()
print("Top 5 tags:", top_tags)

# Step 6: One-hot encode top tags using Pandas only
for tag in top_tags:
    local_news_df[f"tag_{tag}"] = tags_lists.apply(lambda x: int(tag in x))

# Step 7: Drop original tags column if no longer needed
local_news_df.drop(columns=['tags'], inplace=True)


In [ ]:
# Combining any content text columns for..

local_news_df['content'] = (
    local_news_df['title'].fillna('') + " " +
    local_news_df['subtitle'].fillna('') + " " +
    local_news_df['content'].fillna('') + " " +
    local_news_df['top_image_caption'].fillna('')
)

# Dropping the other text columns
local_news_df.drop(columns=['title', 'subtitle', 'top_image_caption'], inplace=True)


In [ ]:
# local_news_articles_cleaned as cleaned dataset for EDA & model training usage
output_file_path_two = 'Datasets\local_news_articles_cleaned.csv'


local_news_df.to_csv(
    output_file_path_two, 
    index=False,
    encoding='utf-8'
)

print(f"Data exported successfully to: {output_file_path_two}")



# > Merge both datasets together
# > Vectorize the content of both datasets

# As of this point, both datasets are identical in terms of metadata. Hence, we can now merge and convert the content column into numerical vectors so that the text can be meaningful for the AI Model.

In [ ]:
print("First dataset:")
police_press_df.info()

print("\nSecond Dataset")
local_news_df.info()

merged_dataset = pd.concat([police_press_df, local_news_df], ignore_index=True)

In [ ]:
'''
merged_dataset = pd.concat([police_press_df, local_news_df], ignore_index=True)



merged_dataset_output = 'Datasets\merged_dataset.csv'


merged_dataset.to_csv(
    merged_dataset_output, 
    index=False,
    encoding='utf-8'
)

print(f"Data exported successfully to: {merged_dataset_output}")

'''

# We are to add on top of the given datasets

## 1. Weather / Geographical APIs
- **Open-Meteo**  
  Free weather API with global coverage. Supports forecast, historical data, and climate data.  
  [https://open-meteo.com](https://open-meteo.com)

- **Copernicus Climate Data Source**  
  Provides free access to historical climate data (temperature, precipitation, etc.) globally.  
  [https://climate.copernicus.eu](https://climate.copernicus.eu)

---

### Free Traffic / Rush Hour Data
- **Open Traffic / OTv2**  
  Open-source platform for historical and real-time traffic/mobility data. Useful to infer rush-hour patterns.  
  [https://github.com/opentraffic/otv2-platform](https://github.com/opentraffic/otv2-platform)

- **GTFS (Static GTFS)**  
  Provides transit schedules, useful to infer expected peak travel periods (rush hours).  
  [https://developers.google.com/transit/gtfs](https://developers.google.com/transit/gtfs)

- **TomTom Traffic API (Free Tier)**  
  Limited free tier for traffic flow and congestion data. Can be used to estimate rush hours.  
  [https://developer.tomtom.com/traffic-api](https://developer.tomtom.com/traffic-api)

In [ ]:
# Vectorize via TF-IDF
# Took advice from Google Gemini which sort of numerical vectorization works best for our given scenario.

vectorizer = TfidfVectorizer(
    max_features=5000,  # only top 5000 words/phrases
    ngram_range=(1,2),  # Consider both singular and pair words
    stop_words='english' # remove common words
)

X_tfidf = vectorizer.fit_transform(merged_dataset['content'])

print("Shape of TF-IDF matrix:")
print(X_tfidf.shape)
print(X_tfidf[0])         # TF-IDF vector for first article